# 1. 2D floor planning with rotation: minimise the area

Lecture: *Week 11 - Floor Planning*, from the *Objective* slide to the second *Solution* slide (PDF pages 17–23).

**The problem.** Six rectangular items must be placed on a floor without overlapping, and each may be turned through 90°. Find the smallest **square** that holds them all. The model minimises the side `v` of that square, which the slides call $A^*$; the square's area is $v^2$, so the smallest side gives the smallest area.

**How to run it.** Run the cells in order, starting with the first: it installs Pyomo and the HiGHS solver (allow a minute the first time). The solve itself takes a few seconds.

**Expected result.** `v= 12.0`: the six items fit in a 12 × 12 square (area 144; the items themselves cover 124). On a desktop HiGHS may print `v= 11.999999` and positions such as `4.999999`: that is rounding caused by the large `M = 9999`, and the answer is still 12.

*Original script: `5 Source Files/Python/model.py` (GLPK solver). This notebook uses HiGHS, which gives the same minimum side, 12. Several layouts reach it, so the positions and rotations drawn may differ from the slides and between computers; `v` is the same.*

In [ ]:
# Run this cell first.
# In the browser (JupyterLite) it installs the extra packages into this tab;
# in Colab or Anaconda it installs them with pip (already installed = no change).
%pip install pyomo highspy

import sys
if sys.platform == "emscripten":  # only in the browser (JupyterLite): adapt Pyomo to it
    import contextlib, pyomo.common.tee, pyomo.common.formatting as _fmt
    # 1) browsers have no threads: skip Pyomo's threaded capture of solver output
    pyomo.common.tee.capture_output = lambda *args, **kwargs: contextlib.nullcontext()
    for _m in ("pyomo.contrib.appsi.solvers.highs", "pyomo.contrib.solver.solvers.highs"):
        if _m in sys.modules:
            sys.modules[_m].capture_output = pyomo.common.tee.capture_output
    # 2) the notebook's output returns None from write(); model.pprint()/display() need a number
    if not getattr(_fmt.StreamIndenter, "_browser_patched", False):
        class _Counted:
            def __init__(self, s): self._s = s
            def write(self, data): self._s.write(data); return len(data)
            def __getattr__(self, name): return getattr(self._s, name)
        def _init(self, ostream, indent=" " * 4, _orig=_fmt.StreamIndenter.__init__):
            _orig(self, ostream, indent)
            if not isinstance(self.target_os, _Counted):
                object.__setattr__(self, "target_os", _Counted(self.target_os))
        _fmt.StreamIndenter.__init__ = _init
        _fmt.StreamIndenter._browser_patched = True

## Load the libraries

Pyomo builds and solves the model, NumPy holds the data and Matplotlib draws the layout at the end. `comb` counts pairs of items; `cm` is not used (the colours come from `plt.get_cmap`).

In [ ]:
from pyomo.environ import *
import numpy as np
from scipy.special import comb      # comb(n, 2) = number of pairs of n items
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.cm as cm          # not used: the colours come from plt.get_cmap below

## The data

The width `w` and height `h` of each item. Items 1–3 are the tables $t_1$–$t_3$ of the slides; item 4 is 7 × 6 (7 × 7 in the slides' four-table example), and items 5 and 6 are the two extra items in the six-item plot of the first *Solution* slide.

In [ ]:
# Data: width w[i] and height h[i] of items 1-6
w = np.array([4,3,6,7,3,2])
h = np.array([5,7,4,6,3,4])

`M` is the "sufficiently large number" of the *Binary Variables* slide: adding `M` to the right-hand side of a constraint switches it off. The two commented lines are the slides' four tables (see *Try it* at the end). `no_binary` is the number of pairs of items, $\binom{6}{2} = 15$; it is not used later.

In [ ]:
# w = np.array([4,3,6,7])
# h = np.array([5,7,4,7])

M = 9999   # big-M: large enough to switch a constraint off

no_items = len(w)
no_binary = int(comb(no_items,2))   # number of pairs i < j (15 for 6 items); not used below

## The model

Create an empty Pyomo model; every set, variable and constraint below is attached to it.

In [ ]:
# Model
model = ConcreteModel()

The items are numbered 1 to 6. `I` and `J` are the same set twice, so that the constraints can loop over pairs of items $(i, j)$.

In [ ]:
# Sets
model.I = RangeSet(no_items)
model.J = RangeSet(no_items)

The sizes become parameters `w[i]` and `h[i]`. Pyomo numbers the items from 1 but NumPy arrays start at 0, hence `w[i-1]`.

In [ ]:
# Parameters
model.w = Param(model.I, initialize=lambda model, i: w[i-1])
model.h = Param(model.I, initialize=lambda model, i: h[i-1])

The variables (slide *MILP Formulation*, with rotation added):

- `x[i]`, `y[i]`: the bottom-left corner of item $i$;
- `v`: the side of the square ($A^*$ on the slides);
- `xij[i, j]`, `yij[i, j]`: the two binary variables of each pair, which choose the one non-overlap constraint that must hold. They are declared for every $i, j$, but only $i < j$ are used;
- `r[i]`: 1 if item $i$ is rotated, 0 if not.

In [ ]:
# Variables
model.x = Var(model.I, within=NonNegativeReals)        # bottom-left corner of item i
model.y = Var(model.I, within=NonNegativeReals)
model.v = Var(within=NonNegativeReals)                 # side of the square (A* on the slides)
model.xij = Var(model.I, model.J, within=Binary)       # the two binaries of each pair choose which
model.yij = Var(model.I, model.J, within=Binary)       # non-overlap constraint must hold (i < j used)
model.r = Var(model.I, within=Binary)                  # 1 = item i rotated by 90 degrees

The objective: minimise `v`. Every item must lie inside $[0, v] \times [0, v]$ (next cell), so this minimises the largest coordinate that any item reaches: a *minimax* problem.

In [ ]:
# Objective
model.obj = Objective(expr=model.v, sense=minimize)

## The constraints

After rotation, item $i$ is $(1-r_i)\,w_i + r_i\,h_i$ wide and $(1-r_i)\,h_i + r_i\,w_i$ high (slide *Amending the constraints*). For every pair $i < j$ there are four non-overlap constraints. The values of $(x_{ij}, y_{ij})$ choose the one that is enforced; the other three get $+M$ or $+2M$ and always hold:

| $(x_{ij}, y_{ij})$ | Constraint enforced | Meaning |
| --- | --- | --- |
| (0, 0) | `overlap_xij` | item $i$ is left of item $j$ |
| (1, 0) | `overlap_xji` | item $j$ is left of item $i$ |
| (0, 1) | `overlap_yij` | item $i$ is below item $j$ |
| (1, 1) | `overlap_yji` | item $j$ is below item $i$ |

The last two rules are the minimax constraints: each item must end at or before `v` in both directions. `Constraint.Skip` leaves out the pairs with $i \ge j$, so each pair is checked once.

In [ ]:
# Constraints
# Width of item i after rotation: (1-r[i])*w[i] + r[i]*h[i]; height: (1-r[i])*h[i] + r[i]*w[i].

# (xij, yij) = (0, 0): item i is LEFT of item j (the other three choices add M or 2M: switched off)
def overlap_xij_rule(model, i, j):
    if i < j:
        return model.x[i] + (1-model.r[i])*model.w[i] + model.r[i]*model.h[i] <= model.x[j] + M*(model.xij[i,j]+model.yij[i,j])
    
    return Constraint.Skip
    
model.overlap_xij = Constraint(model.I, model.J, rule=overlap_xij_rule)

# (xij, yij) = (1, 0): item j is LEFT of item i
def overlap_xji_rule(model, i, j):
    if i < j:
        return model.x[j] + (1-model.r[j])*model.w[j] + model.r[j]*model.h[j] <= model.x[i] + M*(1-model.xij[i,j]+model.yij[i,j])
    
    return Constraint.Skip
model.overlap_xji = Constraint(model.I, model.J, rule=overlap_xji_rule)

# (xij, yij) = (0, 1): item i is BELOW item j
def overlap_yij_rule(model, i, j):
    if i < j:
        return model.y[i] + (1-model.r[i])*model.h[i] + model.r[i]*model.w[i] <= model.y[j] + M*(1+model.xij[i,j]-model.yij[i,j])
        
    return Constraint.Skip

model.overlap_yij = Constraint(model.I, model.J, rule=overlap_yij_rule)

# (xij, yij) = (1, 1): item j is BELOW item i
def overlap_yji_rule(model, i, j):
    if i < j:
        return model.y[j] + (1-model.r[j])*model.h[j] + model.r[j]*model.w[j] <= model.y[i] + M*(2-model.xij[i,j]-model.yij[i,j])
    
    return Constraint.Skip
model.overlap_yji = Constraint(model.I, model.J, rule=overlap_yji_rule)

# minimax constraints: every item must end inside the square [0, v] x [0, v]
def box_xi_rule(model, i):
    return model.x[i] + (1-model.r[i])*model.w[i] + model.r[i]*model.h[i] <= model.v
model.box_xi = Constraint(model.I, rule=box_xi_rule)

def box_yi_rule(model, i):
    return model.y[i] + (1-model.r[i])*model.h[i] + model.r[i]*model.w[i] <= model.v
model.box_yi = Constraint(model.I, rule=box_yi_rule)

A rule that switches rotation off. It is only defined here; the next cell uses it if you delete the `#` in front of `model.rotation_off` (see *Try it*).

In [ ]:
# used to switch off rotation: r[i] <= 0 forces r[i] = 0
def rotation_off(model, i):
    return model.r[i] <= 0

## Solve

HiGHS solves the model; `tee=True` prints its log. `model.display()` then prints the value of every variable and constraint, and `model.pprint()` the whole model. Both are long: scroll past them to the results. Item 5 is a 3 × 3 square, so rotating it changes nothing and `r[5]` has no effect (with GLPK the original script printed `None` for it).

In [ ]:
# uncomment this to switch off rotation
#model.rotation_off = Constraint(model.I, rule=rotation_off)

# Solve
solver = SolverFactory('appsi_highs')
solver.solve(model, tee=True)

model.display()
model.pprint()

## Results

Print each item's corner and whether it is rotated, then draw the layout. A rotated item is drawn $h_i$ wide and $w_i$ high. The `rotated` / `not rotated` lines are in item order, 1 to 6. A value printed as `-0.0` is 0.

In [ ]:
# Print the value of the variables at the optimum
for i in model.I:
    print("x[",i,"]=", model.x[i].value)
    print("y[",i,"]=", model.y[i].value)
    print("r[",i,"]=", model.r[i].value)
print("v=", model.v.value)

fig, ax = plt.subplots()

cmap = plt.get_cmap('Pastel1')  # Can be any colormap available in matplotlib

for i in range(1, no_items+1):
    x = model.x[i].value
    y = model.y[i].value
    wi = w[i-1]
    hi = h[i-1]
    color = cmap(i/no_items)  # Get the color from the colormap
    if model.r[i].value:   # rotated: draw it h wide and w high
        print('rotated')
        rect = patches.Rectangle((x,y),hi,wi,linewidth=2,edgecolor='k',facecolor=color)
    else:
        print('not rotated')
        rect = patches.Rectangle((x,y),wi,hi,linewidth=2,edgecolor='k',facecolor=color)
    ax.add_patch(rect)

plt.grid(True)
plt.axis('equal')
plt.legend([str(x) for x in range(1, no_items+1)], loc='lower left')
plt.xlabel('x')
plt.ylabel('y')
plt.show()

## What do you notice?

The six items cover $20 + 21 + 24 + 42 + 9 + 8 = 124$ square units, but the square has $12 \times 12 = 144$. Could a cleverer layout fit them into an 11 × 11 square?

<details><summary>Answer</summary>

No. An 11 × 11 square has area 121, less than the 124 the items need, so the side must be at least $\sqrt{124} \approx 11.14$. The solver goes further and proves that no layout does better than 12, so 20 square units of the 12 × 12 square stay empty whatever we do. Area gives a lower bound; only the model can tell whether the shapes actually fit.
</details>

## Try it

1. **Rotation off.** In the *Solve* cell, delete the `#` in front of `model.rotation_off = ...`, then choose *Run → Run All Cells*. What is `v` now?
2. **The slides' four tables.** In the data cells, put `#` in front of the two six-item lines (`w = ...`, `h = ...`) and delete it from the two four-item lines, then *Run → Run All Cells*, once with rotation and once without. Compare with the two *Solution* slides.
3. **Mums Know Best!** Use `w = np.array([4,3,3,3,2,2,2,2])` and `h = np.array([1,1,1,1,2,1,1,1])`, with rotation. How large is the square, and how long does the solve take?

<details><summary>Answers</summary>

1. `v = 12` (about a second in the browser). For these six items rotation does not help: the best square is 12 × 12 either way.
2. With rotation `v = 11`, as on the second *Solution* slide; without rotation `v = 12`, as on the first. The four tables cover 114 square units, so area alone would allow $\sqrt{114} \approx 10.7$.
3. `v = 5`: the eight pieces (area 23) fit into a 5 × 5 square with 2 square units empty, as in the plot on the *Mums Know Best!* slide. It takes about 10 seconds in the browser; the slide says MATLAB needed about 10 minutes. Times depend on the solver and the computer.
</details>